## Pipeline di caricamento dati su SQL
In questo notebook configuriamo una pipeline per il caricamento massivo dei dati nel database MariaDB. Iniziamo importando le librerie necessarie per la manipolazione dei dati (`pandas`) e per la gestione della connessione al database (`sqlalchemy`).



In [1]:
import pandas as pd
from sqlalchemy import create_engine

### Caricamento del Dataset Pulito
Ricarichiamo il file CSV precedentemente pulito e processato nella fase di EDA. Questo file contiene tutte le informazioni necessarie, ma è ancora in formato "flat" (piatto), privo degli identificativi numerici necessari per lo schema relazionale SQL.

In [5]:
url= r"C:\Users\angel\OneDrive\Desktop\Lavoro STAFF\hotel_reviews_clean.csv"
hr1= pd.read_csv(url)
engine = create_engine('mysql+mysqlconnector://root:@localhost/Hotel_Reviews')

### Popolamento delle Anagrafiche (Hotels e Reviewers)

In questa fase iniziamo a popolare le tabelle "anagrafiche" del database, che fungeranno da tabelle padre per le relazioni successive.

### Logica dell'operazione:
1. **Estrazione Dati Univoci**: Utilizziamo il metodo `.drop_duplicates()` per assicurarci di caricare ogni Hotel e ogni Nazionalità una sola volta. Questo garantisce l'integrità dei dati ed evita errori di duplicazione sulle Primary Keys.
2. **Rinomina Colonne**: Uniformiamo i nomi delle colonne di Pandas a quelli definiti nello schema SQL (es. da `Hotel_Name` a `hotel_name`).
3. **Caricamento Massivo**: Tramite la funzione `to_sql`, inviamo i dati direttamente al database MariaDB.

Questa procedura è fondamentale per generare gli ID numerici univoci (Auto-increment) che useremo in seguito per collegare le recensioni e le statistiche.


In [7]:
# POPOLAMENTO TABELLA HOTELS 
hotels = hr1[['Hotel_Name', 'Hotel_Address', 'lat', 'lng']].drop_duplicates()
hotels.columns = ['hotel_name', 'hotel_address', 'lat', 'lng']
hotels.to_sql('hotels', con=engine, if_exists='append', index=False)

# POPOLAMENTO TABELLA REVIEWERS 
# Prendiamo le nazionalità univoche
reviewers = hr1[['Reviewer_Nationality']].drop_duplicates()
reviewers.columns = ['reviewer_nationality']
reviewers.to_sql('reviewers', con=engine, if_exists='append', index=False)

print("Tabelle Hotels e Reviewers popolate.")

Tabelle Hotels e Reviewers popolate.


### Connessione al Database e recupero delle Primary Keys
Stabiliamo la connessione con il database locale tramite l'engine di SQLAlchemy. Recuperiamo le tabelle `hotels` e `reviewers` per ottenere gli ID univoci (Primary Keys) generati da SQL. Questi ID sono fondamentali per popolare correttamente la tabella delle recensioni rispettando i vincoli di integrità referenziale.


In [8]:
sql_hotels = pd.read_sql("SELECT hotel_id, hotel_name FROM hotels", engine)
sql_reviewers = pd.read_sql("SELECT reviewer_id, reviewer_nationality FROM reviewers", engine)


### Popolamento della tabella Hotel_Stats (Gestione Duplicati)

In questa fase popoliamo la tabella delle statistiche aggregate. Poiché la tabella `hotel_stats` utilizza `hotel_id` come **Chiave Primaria**, è fondamentale che ogni hotel compaia esattamente una sola volta.

### Passaggi chiave:
1. **Mapping degli ID**: Colleghiamo i nomi degli hotel presenti nel dataset agli ID univoci generati precedentemente nella tabella `hotels`.
2. **Aggregazione e Pulizia**: Il dataset originale può presentare micro-variazioni (es. punteggi leggermente diversi per lo stesso hotel in righe differenti). Utilizziamo `.groupby('hotel_id').agg('max')` per consolidare queste informazioni in un'unica riga coerente per ogni hotel.
3. **Integrità Referenziale**: Questa operazione previene l'errore di "Duplicate Entry" su SQL e garantisce la corretta relazione 1:1 tra l'anagrafica dell'hotel e le sue statistiche globali.


In [10]:
# POPOLAMENTO TABELLA HOTEL_STATS 
# 1. Filtriamo solo le colonne necessarie
stats = hr1[['Hotel_Name', 'Average_Score', 'Total_Number_of_Reviews', 'Additional_Number_of_Scoring']]

# 2. Colleghiamo gli ID degli hotel
stats = stats.merge(sql_hotels, left_on='Hotel_Name', right_on='hotel_name')

# 3. Gestiamo i duplicati aggregando: prendiamo il valore massimo per ogni ID hotel
# Questo risolve eventuali errori della chiave primaria
stats = stats.groupby('hotel_id').agg({
    'Average_Score': 'max',
    'Total_Number_of_Reviews': 'max',
    'Additional_Number_of_Scoring': 'max'
}).reset_index()

# 4. Rinominiamo le colonne per SQL
stats.columns = ['hotel_id', 'average_score', 'total_reviews', 'additional_number_of_scoring']

# 5. Carichiamo
stats.to_sql('hotel_stats', con=engine, if_exists='append', index=False)

print("Tabella Hotel_Stats popolata correttamente senza duplicati.")


Tabella Hotel_Stats popolata correttamente senza duplicati.


## Caricamento Massivo della Tabella 'Reviews' (Bulk Insert)

Questo passaggio finale è dedicato esclusivamente al popolamento della tabella `reviews`. Data la presenza di oltre 511.000 record e la complessità dei testi contenuti nelle recensioni, ho deciso di  implementare una strategia di caricamento ottimizzata.

### Dettagli tecnici del processo:
1. **Mapping Finale degli ID**: Colleghiamo l'intero dataset originale agli ID univoci degli Hotel e dei Recensori per stabilire le relazioni (Foreign Keys) richieste dallo schema SQL.
2. **Selezione e Formattazione**: Filtriamo solo le colonne necessarie e le rinominiamo per farle coincidere esattamente con la struttura della tabella di destinazione.
3. **Tecnica di Chunking**: Utilizziamo il parametro `chunksize=10000` per inviare i dati a blocchi. Questo previene il sovraccarico della memoria RAM e scongiura errori di timeout del database MariaDB, garantendo un caricamento fluido ed efficiente.

Conclusa questa fase, l'intero ecosistema relazionale del database è popolato e pronto per le interrogazioni analitiche.



In [11]:
# Uniamo gli ID al dataframe originale per creare i collegamenti (Foreign Keys)
hr_final = hr1.merge(sql_hotels, left_on='Hotel_Name', right_on='hotel_name')
hr_final = hr_final.merge(sql_reviewers, left_on='Reviewer_Nationality', right_on='reviewer_nationality')

# Selezioniamo e rinominiamo le colonne per farle combaciare con lo schema SQL
reviews_table = hr_final[[
    'hotel_id', 'reviewer_id', 'Review_Date', 'Reviewer_Score', 
    'Negative_Review', 'Positive_Review', 'Review_Total_Negative_Word_Counts', 
    'Review_Total_Positive_Word_Counts', 'Total_Number_of_Reviews_Reviewer_Has_Given',
    'days_since_review', 'Tags', 'Review_Year', 'Review_Month', 
    'Days_Since_Numeric', 'Total_Review_Length'
]]

reviews_table.columns = [
    'hotel_id', 'reviewer_id', 'review_date', 'reviewer_score', 
    'negative_review', 'positive_review', 'review_total_negative_word_counts', 
    'review_total_positive_word_counts', 'total_reviews_reviewer_has_given',
    'days_since_review', 'tags', 'review_year', 'review_month', 
    'days_since_numeric', 'total_review_length'
]

# Caricamento massivo (circa 515.000 righe)
reviews_table.to_sql('reviews', con=engine, if_exists='append', index=False, chunksize=10000)

print("Pipeline conclusa con successo! Tutti i dati sono su SQL.")


Pipeline conclusa con successo! Tutti i dati sono su SQL.
